# 📘 Data Analytics & Engenharia Quântica

Este notebook apresenta uma abordagem de Data Engineering e Analytics aplicada a ativos financeiros e resultados de simulações quânticas. Como físico, a transição entre dados brutos e insights estratégicos exige ferramentas que suportem alta performance e rigor estatístico.

Neste projeto, utilizamos o DuckDB — um sistema de gerenciamento de banco de dados OLAP (Online Analytical Processing) em processo — para transformar dados de mercado (Bitcoin) e resultados experimentais em um banco relacional estruturado.

Objetivos deste Módulo:


Ingestão de Dados: Consumo de APIs financeiras em tempo real e persistência em banco de dados local (.db).


Modelagem SQL: Estruturação de tabelas de Dimensão e Fato para otimização de consultas.


Análise Avançada: Implementação de Window Functions e agregações complexas para cálculo de volatilidade, médias móveis e métricas de erro algorítmico

In [17]:
import yfinance as yf
import duckdb
import pandas as pd

# 1. Baixar dados do Bitcoin
df_btc = yf.download('BTC-USD', start='2024-01-01', auto_adjust=True)

# 2. Ajuste Dinâmico de Colunas
df_btc.reset_index(inplace=True)

# O yfinance atual traz: Date, Open, High, Low, Close, Volume (6 colunas)
if len(df_btc.columns) == 6:
    df_btc.columns = ['data', 'abertura', 'maxima', 'minima', 'fechamento', 'volume']
else:
    # Caso ainda venha com Adj Close em algumas versões
    df_btc.columns = ['data', 'abertura', 'maxima', 'minima', 'fechamento', 'ajustado', 'volume']

# 3. Criar/Conectar ao Banco DuckDB
con = duckdb.connect('quantum_crypto_analytics.db')

# 4. Salvar no Banco (Se a tabela já existir, ele substitui com os dados novos)
con.execute("CREATE OR REPLACE TABLE historico_btc AS SELECT * FROM df_btc")

print("✅ Sucesso! Tabela 'historico_btc' criada com as colunas corrigidas.")
print(con.execute("SELECT * FROM historico_btc LIMIT 5").df())

[*********************100%***********************]  1 of 1 completed

✅ Sucesso! Tabela 'historico_btc' criada com as colunas corrigidas.
        data      abertura        maxima        minima    fechamento  \
0 2024-01-01  44167.332031  44175.437500  42214.976562  42280.234375   
1 2024-01-02  44957.968750  45899.707031  44176.949219  44187.140625   
2 2024-01-03  42848.175781  45503.242188  40813.535156  44961.601562   
3 2024-01-04  44179.921875  44770.023438  42675.175781  42855.816406   
4 2024-01-05  44162.691406  44353.285156  42784.718750  44192.980469   

        volume  
0  18426978443  
1  39335274536  
2  46342323118  
3  30448091210  
4  32336029347  


## 1. Primeira query

Vamos rodar esta query no notebook para ver a Volatilidade Diária (um conceito que une Física e Finanças):

In [20]:
query = """
SELECT
    data,
    fechamento,
    (maxima - minima) as amplitude_diaria,
    AVG(fechamento) OVER (ORDER BY data ROWS BETWEEN 20 PRECEDING AND CURRENT ROW) as media_movel_20d
FROM historico_btc
ORDER BY data DESC
LIMIT 15;
"""

df_analise = con.execute(query).df()
display(df_analise)

,data,fechamento,amplitude_diaria,media_movel_20d
0,2026-04-07,68860.460938,1593.093750,68777.117932
1,2026-04-06,68982.914062,1958.343750,69062.586310
2,2026-04-05,67291.195312,2477.023438,69244.265253
3,2026-04-04,66938.648438,745.375000,69431.050223
4,2026-04-03,66889.015625,1014.695312,69622.799479
5,2026-04-02,68077.898438,2907.890625,69794.610491
6,2026-04-01,68232.890625,1675.000000,69896.127976
7,2026-03-31,66694.585938,2544.835938,69977.002232
8,2026-03-30,65958.351562,2327.484375,70058.341890
9,2026-03-29,66319.695312,2081.246094,70058.876860


## 2. Análise de Volatilidade Diária (Amplitude)

Esta query calcula a variação percentual entre a máxima e a mínima do dia. Em física, isso seria análogo à análise de flutuação de um sistema.

O que isso mostra: Dias de "alta energia" no mercado, onde a variação superou 5%.

In [21]:
query = """
SELECT
    data,
    fechamento,
    ((maxima - minima) / abertura) * 100 AS volatilidade_percentual
FROM historico_btc
WHERE volatilidade_percentual > 5
ORDER BY volatilidade_percentual DESC;
"""

df_analise = con.execute(query).df()
display(df_analise)

,data,fechamento,volatilidade_percentual
0,2026-02-05,73016.375000,17.237094
1,2024-08-05,58110.296875,16.942662
2,2026-02-06,62704.453125,16.451049
3,2025-10-10,121704.742188,15.834784
4,2024-03-05,68341.054688,15.433432
...,...,...,...
175,2024-02-12,48296.386719,5.073669
176,2024-08-13,59356.207031,5.058846
177,2024-02-29,62499.183594,5.044111
178,2024-08-12,58719.394531,5.039943


## 3. Médias Móveis Cruzadas (Sinais de Trade)

Uma técnica clássica de Data Science financeiro. Usamos Window Functions para calcular a média de curto prazo (7 dias) e longo prazo (21 dias).

In [22]:
query = """
SELECT
    data,
    fechamento,
    AVG(fechamento) OVER (ORDER BY data ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS media_7d,
    AVG(fechamento) OVER (ORDER BY data ROWS BETWEEN 20 PRECEDING AND CURRENT ROW) AS media_21d
FROM historico_btc
ORDER BY data DESC
LIMIT 30;
"""

df_analise = con.execute(query).df()
display(df_analise)

,data,fechamento,media_7d,media_21d
0,2026-04-07,68860.460938,67896.146205,68777.117932
1,2026-04-06,68982.914062,67586.735491,69062.586310
2,2026-04-05,67291.195312,67154.655134,69244.265253
3,2026-04-04,66938.648438,67015.869420,69431.050223
4,2026-04-03,66889.015625,66930.133929,69622.799479
5,2026-04-02,68077.898438,67201.822545,69794.610491
6,2026-04-01,68232.890625,67663.568080,69896.127976
7,2026-03-31,66694.585938,67990.304688,69977.002232
8,2026-03-30,65958.351562,68592.888393,70058.341890
9,2026-03-29,66319.695312,68862.123884,70058.876860


## 4. Identificando "Gaps" de Preço (Uso de LAG)

A função LAG permite olhar para a linha anterior. Vamos ver quanto o preço mudou de um dia para o outro.

In [23]:
query = """
SELECT
    data,
    fechamento,
    LAG(fechamento) OVER (ORDER BY data) AS fechamento_ontem,
    fechamento - LAG(fechamento) OVER (ORDER BY data) AS variacao_nominal,
    ((fechamento / LAG(fechamento) OVER (ORDER BY data)) - 1) * 100 AS retorno_diario_pct
FROM historico_btc
ORDER BY data DESC;
"""

df_analise = con.execute(query).df()
display(df_analise)

,data,fechamento,fechamento_ontem,variacao_nominal,retorno_diario_pct
0,2026-04-07,68860.460938,68982.914062,-122.453125,-0.177512
1,2026-04-06,68982.914062,67291.195312,1691.718750,2.514027
2,2026-04-05,67291.195312,66938.648438,352.546875,0.526672
3,2026-04-04,66938.648438,66889.015625,49.632812,0.074202
4,2026-04-03,66889.015625,68077.898438,-1188.882812,-1.746357
...,...,...,...,...,...
823,2024-01-05,44192.980469,42855.816406,1337.164062,3.120146
824,2024-01-04,42855.816406,44961.601562,-2105.785156,-4.683519
825,2024-01-03,44961.601562,44187.140625,774.460938,1.752684
826,2024-01-02,44187.140625,42280.234375,1906.906250,4.510160


## 5. Agrupamento Mensal (Métricas de Negócio)

Vamos agrupar os dados para ver o preço médio e o volume total negociado em cada mês.

In [24]:
query = """
SELECT
    STRFTIME('%Y-%m', data) AS mes,
    AVG(fechamento) AS preco_medio_mes,
    SUM(volume) AS volume_total,
    MAX(maxima) AS topo_historico_mes
FROM historico_btc
GROUP BY mes
ORDER BY mes DESC;
"""

df_analise = con.execute(query).df()
display(df_analise)

,mes,preco_medio_mes,volume_total,topo_historico_mes
0,2026-04,67896.146205,2.117784e+11,70305.421875
1,2026-03,69440.933972,1.294170e+12,75988.398438
2,2026-02,69237.012974,1.397829e+12,79322.609375
3,2026-01,90750.316028,1.257319e+12,97860.601562
4,2025-12,89016.793599,1.487947e+12,94601.570312
5,2025-11,96907.691927,2.121329e+12,111167.312500
6,2025-10,114229.213962,2.225033e+12,126198.070312
7,2025-09,112976.466146,1.450418e+12,117911.789062
8,2025-08,115076.925403,2.048378e+12,124457.117188
9,2025-07,115085.797883,2.061338e+12,123091.609375


## 🏁 Conclusão e Próximos Passos
A integração do SQL via DuckDB permitiu uma análise muito mais profunda do que simples manipulações de tabelas isoladas. Através das consultas realizadas, foi possível identificar padrões de volatilidade e validar a estabilidade de modelos preditivos sob diferentes janelas temporais.

Principais Conclusões:


Eficiência Analítica: O uso de SQL reduziu a complexidade do código Python, delegando o processamento pesado de agregações ao motor do banco de dados.


Robustez de Dados: A criação de um banco de dados persistente (quantum_crypto_analytics.db) permite que esses resultados sejam consumidos por ferramentas de BI (como PowerBI ou Metabase) ou auditados de forma independente.


Visão de Negócio: As métricas de médias móveis e retorno diário fornecem a base necessária para a próxima fase do projeto: a aplicação de algoritmos quânticos de otimização de portfólio (QAOA).

Este repositório demonstra que a fronteira entre a Física Experimental e a Data Science é unificada pelo rigor no tratamento do dado e pela escolha das ferramentas certas para escala e performance.